<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания №11


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Создать базовый класс Customer в C#, который будет представлять информацию о 
клиентах или покупателях. На основе этого класса разработать 2-3 производных 
класса, демонстрирующих принципы наследования и полиморфизма. В каждом из 
классов  должны  быть  реализованы  новые  атрибуты  и  методы,  а  также 
переопределены  некоторые  методы  базового  класса  для  демонстрации 
полиморфизма.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) исользуйтие в проекте коллекции, делегаты, события.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [2]:
//  ДЕЛЕГАТ ДЛЯ ОТЧЁТОВ
public delegate void CustomerReport(Customer c);

// БАЗОВЫЙ КЛАСС
public abstract class Customer
{
    public int CustomerId { get; set; }
    public string Name { get; private set; }          
    public string Email { get; private set; }         
    public string Phone { get; set; }                 
    public string Address { get; set; }               
    public bool IsActive { get; private set; } = true;
    public DateTime CreatedAt { get; } = DateTime.Now;

    // Событие базового класса (уведомления об изменениях)
    public event EventHandler<string> CustomerChanged;

    protected Customer(string name, string email)
    {
        Name = name;
        Email = email;
    }

    protected void OnChanged(string message) =>
        CustomerChanged?.Invoke(this, message);

    public virtual string GetFullName() => Name;

    public virtual void UpdateEmail(string newEmail)
    {
        Email = newEmail;
        OnChanged($"Email изменён на {Email}");
    }

    public virtual void ViewProfile()
    {
        Console.WriteLine($"ID: {CustomerId}, Имя: {Name}, Email: {Email}, Тел.: {Phone}, Адрес: {Address}, Активен: {IsActive}, Создан: {CreatedAt:g}");
        OnChanged("Профиль просмотрен");
    }

    public void Activate()  { IsActive = true;  OnChanged("Аккаунт активирован"); }
    public void Deactivate(){ IsActive = false; OnChanged("Аккаунт деактивирован"); }
    public void ChangeName(string newName) { Name = newName; OnChanged($"Имя изменено на {Name}"); }

    public virtual decimal CalculateDiscount(decimal orderTotal) => 0m;
}

public class VipCustomer : Customer
{
    public int LoyaltyPoints { get; private set; }
    public decimal DiscountRate { get; set; } = 10m;   
    public DateTime VipStartDate { get; set; } = DateTime.Today;
    public string Tier { get; private set; } = "Silver";

    public VipCustomer(string name, string email) : base(name, email) {}

    public void AddPoints(int pts) { LoyaltyPoints += pts; OnChanged($"+{pts} баллов, всего {LoyaltyPoints}"); }
    public bool RedeemPoints(int pts)
    {
        if (LoyaltyPoints < pts) return false;
        LoyaltyPoints -= pts; OnChanged($"Списано {pts} баллов, осталось {LoyaltyPoints}");
        return true;
    }
    public void UpgradeTier(string newTier) { Tier = newTier; OnChanged($"Статус повышен до {Tier}"); }

    public override void ViewProfile()
    {
        Console.WriteLine($"[VIP] ID:{CustomerId}, Имя:{GetFullName()}, Email:{Email}, Баллы:{LoyaltyPoints}, Скидка:{DiscountRate}%, Tier:{Tier}, VIP с:{VipStartDate:d}");
        OnChanged("Профиль VIP просмотрен");
    }

    public override decimal CalculateDiscount(decimal orderTotal)
        => Math.Round(orderTotal * (DiscountRate / 100m) + Math.Min(LoyaltyPoints, 500) * 0.01m, 2);
}

public class RegularCustomer : Customer
{
    public DateTime RegistrationDate { get; set; } = DateTime.Today;
    public DateTime LastEmailUpdate { get; private set; }
    public int OrdersCount { get; private set; }
    public string PreferredLanguage { get; set; } = "ru";

    public RegularCustomer(string name, string email) : base(name, email) {}

    public void PlaceOrder()
    {
        OrdersCount++;
        OnChanged($"Оформлён заказ. Всего заказов: {OrdersCount}");
    }

    public void Login()
    {
        OnChanged("Пользователь вошёл в систему");
    }

    public override void UpdateEmail(string newEmail)
    {
        base.UpdateEmail(newEmail);
        LastEmailUpdate = DateTime.Now;
    }

    public override void ViewProfile()
    {
        Console.WriteLine(
            $"[REG] ID:{CustomerId}, Имя:{GetFullName()}, Email:{Email}, Заказы:{OrdersCount}, " +
            $"Регистрация:{RegistrationDate:d}, LastEmailUpd:{((LastEmailUpdate == default) ? "-" : LastEmailUpdate.ToString("g"))}, " +
            $"Язык:{PreferredLanguage}"
        );
        OnChanged("Профиль REG просмотрен");
    }

    public override decimal CalculateDiscount(decimal orderTotal)
        => OrdersCount >= 5 ? Math.Round(orderTotal * 0.05m, 2) : 0m;
}

public class GroupCustomer : Customer
{
    public string GroupName { get; private set; }
    public int MembersCount { get; private set; }
    public string Coordinator { get; private set; }
    public string Department { get; set; }

    public GroupCustomer(string groupName, string email)
        : base(groupName, email)
    {
        GroupName = groupName;
    }

    public void AddMember(int count = 1) { MembersCount += count; OnChanged($"Добавлено участников: {count}. Всего: {MembersCount}"); }
    public void RemoveMember(int count = 1) { MembersCount = Math.Max(0, MembersCount - count); OnChanged($"Удалено участников: {count}. Всего: {MembersCount}"); }
    public void AssignCoordinator(string name) { Coordinator = name; OnChanged($"Назначен координатор: {Coordinator}"); }

    public override string GetFullName() => GroupName;

    public override void ViewProfile()
    {
        Console.WriteLine($"[GROUP] ID:{CustomerId}, Группа:{GroupName}, Email:{Email}, Участников:{MembersCount}, Координатор:{Coordinator ?? "-"}, Отдел:{Department ?? "-"}");
        OnChanged("Профиль GROUP просмотрен");
    }

    public override decimal CalculateDiscount(decimal orderTotal)
        => MembersCount >= 10 ? Math.Round(orderTotal * 0.08m, 2) : 0m;
}



Console.WriteLine("=== Customers: коллекции, делегаты, события ===\n");

// Коллекция клиентов (List<T>)
var customers = new List<Customer>
{
    new VipCustomer("Никита", "nikita@mail.com") { CustomerId = 1, DiscountRate = 15, Phone = "+7 999 000-11-22" },
    new RegularCustomer("Егор", "egor@mail.com") { CustomerId = 2, RegistrationDate = new DateTime(2022,5,1), Phone = "+7 900 123-45-67" },
    new GroupCustomer("Студенты", "group@mail.com") { CustomerId = 3, Department = "Обучение", Phone = "+7 999 888-77-66" }
};

// Словарь для быстрого доступа (Dictionary<TKey,TValue>)
var index = customers.ToDictionary(c => c.CustomerId, c => c);

// Подписка на события (общий обработчик + индивидуальные)
foreach (var c in customers)
{
    c.CustomerChanged += (sender, msg) =>
    {
        var who = (Customer)sender;
        Console.WriteLine($"[EVENT] #{who.CustomerId} {who.GetFullName()}: {msg}");
    };
}

// Делегат отчёта: сначала печатаем профиль, затем считаем скидку
CustomerReport report = c => c.ViewProfile();
report += c =>
{
    var discount = c.CalculateDiscount(1000m);
    Console.WriteLine($"Скидка на заказ 1000₽: {discount:0.##}₽\n");
};

// Действия над клиентами (вызывают события)
((VipCustomer)index[1]).AddPoints(120);
((VipCustomer)index[1]).UpgradeTier("Gold");

((RegularCustomer)index[2]).Login();
((RegularCustomer)index[2]).PlaceOrder();
index[2].UpdateEmail("egor.new@mail.com");

((GroupCustomer)index[3]).AddMember(12);
((GroupCustomer)index[3]).AssignCoordinator("Анна");

Console.WriteLine();
// Печать отчёта по всем клиентам (делегат)
foreach (var c in customers) report(c);

// Пример лямбда-фильтра: только активные с «выгодной» скидкой
Console.WriteLine("— Активные со скидкой > 50₽ на заказ 1000₽ —");
var winners = customers.Where(c => c.IsActive && c.CalculateDiscount(1000m) > 50m);
foreach (var c in winners) Console.WriteLine($"#{c.CustomerId} {c.GetFullName()} — скидка {c.CalculateDiscount(1000m):0.##}₽");

Console.WriteLine("\n=== Конец ===");


=== Customers: коллекции, делегаты, события ===

[EVENT] #1 Никита: +120 баллов, всего 120
[EVENT] #1 Никита: Статус повышен до Gold
[EVENT] #2 Егор: Пользователь вошёл в систему
[EVENT] #2 Егор: Оформлён заказ. Всего заказов: 1
[EVENT] #2 Егор: Email изменён на egor.new@mail.com
[EVENT] #3 Студенты: Добавлено участников: 12. Всего: 12
[EVENT] #3 Студенты: Назначен координатор: Анна

[VIP] ID:1, Имя:Никита, Email:nikita@mail.com, Баллы:120, Скидка:15%, Tier:Gold, VIP с:10/17/2025
[EVENT] #1 Никита: Профиль VIP просмотрен
Скидка на заказ 1000₽: 151.2₽

[REG] ID:2, Имя:Егор, Email:egor.new@mail.com, Заказы:1, Регистрация:5/1/2022, LastEmailUpd:10/17/2025 9:06 PM, Язык:ru
[EVENT] #2 Егор: Профиль REG просмотрен
Скидка на заказ 1000₽: 0₽

[GROUP] ID:3, Группа:Студенты, Email:group@mail.com, Участников:12, Координатор:Анна, Отдел:Обучение
[EVENT] #3 Студенты: Профиль GROUP просмотрен
Скидка на заказ 1000₽: 80₽

— Активные со скидкой > 50₽ на заказ 1000₽ —
#1 Никита — скидка 151.2₽
#3 Студен